# Phase 3: 通用_模型训练并验证

## 概述 Overview

吸取先前的所有教训，重新构建一个Phase3

主要用于进行模型验证，适用于独立验证和交叉验证，并将结果输出到json上

**模型更新内容如下**
- npz中的ids不能再与csv中的id实现对应了，需要默认按照内容进行一一对齐
- 不再考虑筛选内容（Phase3中仅考虑异常值）
- 由于数据集已经拆分，每次只能做一种表征和一种任务了


**超参数配置：**
- Logistic Regression: L2正则化, balanced权重
- Random Forest: 100棵树, balanced权重
- XGBoost: GPU加速, max_depth=6, balanced权重

**数据配置：**
- 半衰期阈值：SIF：270，SGF：250
- 半衰期异常值：700
- 筛选条件：（在前面已经进行过筛选 ）
- 随机规则：**随机种子**+**每次随机洗牌数据集顺序**

**主要功能:**
- 数据集选择csv，需要手动指定**训练集**以及**测试集**
- 多轮次交叉验证取平均值（可选）
- 多轮次独立验证取平均值（可选）
- 根据mask数组筛选是否需要掩盖或者选择部分特征（可选，只要选择不用的部分就行了）
- 输出一张图，验证当前训练集和测试集的可信程度（包括分布情况，但是仅限于欧氏距离，可能没有实际意义）

**输出**：
- 输出一些数据运行的json，不额外产生其他内容


---

## 1. 环境检查与导入 Environment Setup

**说明：**
- 该部分负责全部的文件导入，一般不要动，所有的Phase都保持一致
- 需要追加非核心库的时候，在对应的cell追加即可

In [14]:
# 环境检查
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
print(project_root)
sys.path.insert(0, str(project_root / "src"))

# 核心库导入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
warnings.filterwarnings('ignore')

# 机器学习
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# 检查GPU可用性
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU可用: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPU不可用，将使用CPU训练")
except ImportError:
    gpu_available = False
    print("⚠ PyTorch未安装，将使用CPU训练")

# 设置显示选项
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ 所有库已成功导入")
print(f"✓ 项目根目录: {project_root}")


# 对象转换的方法，也放在这里

def convert_numpy_types(obj):
    """递归转换numpy类型为Python原生类型"""
    import numpy as np
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


d:\RA\feature_extraction
⚠ GPU不可用，将使用CPU训练
✓ 所有库已成功导入
✓ 项目根目录: d:\RA\feature_extraction


## 2. 参数配置区 Configuration

**⚙️ 根据您的需求修改以下参数**

In [15]:
# ============== 参数配置区 ==============

CONFIG = {

    # -------------------------------------文件读取路径--------------------------------------
    # csv文件读取路径
    'processed_dir': project_root / 'data' / 'general' / 'csv',
    # npz文件读取路径
    'features_dir': project_root / 'data' / 'general' / 'features',
    # 一些可能用不上的（在具体的cell中自行规范输出）
    #'cv_results_dir': project_root / 'data' / 'feature-engineering'/ 'outputs' / 'phase3' / 'cv_results',
    #'feature_importance_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'feature_importance',
    #'transfer_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'transfer_results',
    #'figures_dir': project_root / 'data' / 'feature-engineering'  / 'figures' / 'phase3',
    #'independent_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'independent_results',
    # 一个临时输出的文件夹
    'temp':project_root/'data'/'general'/'outputs',
    # -------------------------------------结果输出------------------------------------------
    # 交叉和独立验证结果输出
    'Final_Results_path':project_root/'data'/'general'/'outputs', 



    # -------------------------------------手动设置区域-----------------------------------------------------
    # 需要执行的任务包括
    'Target':'SGF',
    # 用于验证模型效果的数据名称
    'dataset_names':{
        'train':"Train_sgf_Morgan(1024)_Avalon(512)",
        'test':"Test_sgf_Morgan(1024)_Avalon(512)"
    },
    # message，用于标注输出文件上的信息
    'message':'sgf_Morgan(1024)_Avalon(512)',# Morgan(1024)_Avalon(512)_ChemBERTa(384)

    # 掩码文件的所在位置（要求，特定任务上，无需使用的特征为0）
    'mask':"_error__importance_mask.json",




    # -------------------------------------超参数验证区域--------------------------------------------------

    # 模型选择（可选: 'lr', 'rf', 'xgb'）
    'models_to_train': ['lr', 'rf', 'xgb'],
    
    # 交叉验证参数
    'n_folds': 5,
    'random_state': 42,
    
    # XGBoost参数
    'use_gpu': gpu_available,
    'xgb_max_depth': 6,
    'xgb_learning_rate': 0.1,
    'xgb_n_estimators': 100,
    
    # Random Forest参数
    'rf_n_estimators': 100,
    'rf_n_jobs': -1,
    
    # Logistic Regression参数
    'lr_max_iter': 1000,
    
    # 可视化参数
    'dpi': 300,
    'format': 'png',
    'display_plots': True,
    'max_display_plots': 8,

    # 阈值
    'threshold':{
        'SIF':270,
        'SGF':250
    },

    # 是否只提取单体分子
    'is_monomer': True,


}

CONFIG['Final_Results_json_path']=CONFIG['Final_Results_path'] / f"Final_Results_{CONFIG['message']}.json"

# 创建输出目录
for key in ['temp']:
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print("配置参数:")
print(f"  模型: {CONFIG['models_to_train']}")
print(f"  交叉验证折数: {CONFIG['n_folds']}")
print(f"  GPU加速: {CONFIG['use_gpu']}")
print(f"  XGBoost参数: max_depth={CONFIG['xgb_max_depth']}, lr={CONFIG['xgb_learning_rate']}")

配置参数:
  模型: ['lr', 'rf', 'xgb']
  交叉验证折数: 5
  GPU加速: False
  XGBoost参数: max_depth=6, lr=0.1


## 3. 数据加载与二值化 Data Loading & Binarization

**筛选原则：**
- is_monomer=Ture
- 半衰期不能同时为0
- 数值低于异常值

In [16]:
import numpy as np
import pandas as pd
from pathlib import Path

def load_and_binarize_dataset(npz_path: Path, csv_path: Path, target: str, is_monomer: bool = None):
    """
    加载数据并将标签二值化，采用索引顺序匹配而非ID匹配
    (2026-01-16 更新: 解决ID重复问题，直接按行顺序对应)
    
    Args:
        npz_path: NPZ特征文件 (包含 X, y_sif, y_sgf 等)
        csv_path: 处理后的CSV文件 (包含标签和筛选列)
        target: 'SIF' or 'SGF'
        is_monomer: 如果为True，只保留monomer；False，只保留非monomer；None不筛选
    
    Returns:
        X_valid, y_binary, median, feature_names
    """
    # 1. 加载NPZ特征
    data = np.load(npz_path, allow_pickle=True)
    X = data['X']
    feature_names = data['feature_names']
    # 注意：这里不再提取 ids_npz，因为我们直接按物理顺序读取
    
    # 2. 加载CSV获取标签及筛选条件
    df = pd.read_csv(csv_path)
    
    # 3. 顺序匹配逻辑
    # 既然 npz 的 X 属性与 csv 默认顺序对应，我们直接根据 df 的索引进行筛选
    valid_indices = []
    valid_labels = []
    
    label_col = f"{target}_minutes"
    
    # 使用 zip 或直接遍历索引，确保 X 的第 i 行对应 df 的第 i 行
    for i, row in df.iterrows():
        # 获取当前行的标签
        label = row[label_col]
        
        # =================================== 过滤操作, 虽然已经操作过了 ===================================
        # 1. 标签有效性判断
        if label == -1 or pd.isna(label):
            continue
            
        # 2. monomer 条件筛选
        if is_monomer is not None and row['is_monomer'] != is_monomer:
            continue
            
        # 3. 特殊阈值处理 (SIF/SGF_minutes > 700 排除)
        if label > 700:
            continue
        # ===============================================================================
        
        # 记录通过筛选的行索引和对应的标签
        valid_indices.append(i) # 选择合理的X
        valid_labels.append(label) # 选择合理的y
    
    # 4. 筛选有效样本
    # 利用 numpy 的高级索引，根据保存的行索引一次性提取对应的特征行
    X_valid = X[valid_indices]
    y_minutes = np.array(valid_labels)

    
    
    # 5. 设定阈值并二值化
    # 假设 CONFIG 已经在全局定义，如果没有，请确保在此处能访问到
    median = None
    if target == 'SIF':
        median = CONFIG['threshold']['SIF']
    elif target == 'SGF':
        median = CONFIG['threshold']['SGF']
    else:
        raise ValueError(f"Unknown target: {target}")

    # 1 = 稳定, 0 = 不稳定
    y_binary = (y_minutes >= median).astype(int)
    
    print(f"--- 数据加载完成 ---")
    print(f"  有效样本数: {len(X_valid)}")
    print(f"  任务目标: {target} (中位数阈值: {median:.1f})")
    print(f"  类别分布 (1/0): {np.sum(y_binary == 1)} / {np.sum(y_binary == 0)}")
    
    return X_valid, y_binary, median, feature_names


# 获取数据集名称：要求csv和npz文件是一样的，因为前面要求同样的内容
TRAIN_NAME=CONFIG['dataset_names']['train']
TEST_NAMS=CONFIG['dataset_names']['test']
datasets_data = {}
npz_files = [
    CONFIG['features_dir']/f"{TRAIN_NAME}.npz",
    CONFIG['features_dir']/f"{TEST_NAMS}.npz"
]
is_monomer = CONFIG['is_monomer'] # 仅加载monomer样本，设置为False则加载非monomer样本，None则不筛选
print(f"加载并二值化 {len(npz_files)} 个数据集:\n")
for npz_file in npz_files:
    dataset_name = npz_file.stem.replace('', '')# 这行代码看起来没啥用，其实是历史遗留问题，不用管也不要动
    csv_file = CONFIG['processed_dir'] / f"{dataset_name}.csv"
    if CONFIG['Target']=='SIF':
        X_sif, y_sif, median_sif, feat_names = load_and_binarize_dataset(npz_file, csv_file, 'SIF',is_monomer)
        datasets_data[dataset_name] = {
            'X_sif': X_sif,
            'y_sif': y_sif,
            'median_sif': median_sif,
            'feature_names': feat_names,
    }
    elif CONFIG['Target']=='SGF':
        X_sgf, y_sgf, median_sgf, feat_names= load_and_binarize_dataset(npz_file, csv_file, 'SGF',is_monomer)
        datasets_data[dataset_name] = {
            'X_sgf': X_sgf,
            'y_sgf': y_sgf,
            'median_sgf': median_sgf,
            'feature_names': feat_names,
    }
    else:
        print("请指定任务为SGF或者SIF")

print(f"✓ 数据加载完成！共 {len(datasets_data)} 个数据集")

加载并二值化 2 个数据集:

--- 数据加载完成 ---
  有效样本数: 370
  任务目标: SGF (中位数阈值: 250.0)
  类别分布 (1/0): 101 / 269
--- 数据加载完成 ---
  有效样本数: 32
  任务目标: SGF (中位数阈值: 250.0)
  类别分布 (1/0): 20 / 12
✓ 数据加载完成！共 2 个数据集


## 4. 独立/交叉/等参数选取

**做出改动**
- 更新模型选取的种子，每次均为随机生成
- 每次对数据集重新洗牌，保证每次预测都不一样
- 双管齐下，我还不信了捏

**规范输出结构**

同时输出两个metric和results
```
        metric = {
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'auc': []
        }
```

In [17]:
import random
# 选择模型：这里已经强制了随机种子1-100000
def get_model(model_name: str, use_gpu: bool = False):
    
    # CONFIG['random_state']=random.randint(1, 1000000)
    # print("选择模型为", model_name, "随机种子", CONFIG['random_state'])
    """
    创建模型实例
    """
    if model_name == 'lr':
        return LogisticRegression(
            max_iter=CONFIG['lr_max_iter'],
            class_weight='balanced',
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'rf':
        return RandomForestClassifier(
            n_estimators=CONFIG['rf_n_estimators'],
            class_weight='balanced',
            n_jobs=CONFIG['rf_n_jobs'],
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'xgb':
        params = {
            'max_depth': CONFIG['xgb_max_depth'],
            'learning_rate': CONFIG['xgb_learning_rate'],
            'n_estimators': CONFIG['xgb_n_estimators'],
            'random_state': random.randint(1, 1000000),
            'tree_method': 'hist',
        }
        if use_gpu:
            params['device'] = 'cuda:0'
        return XGBClassifier(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")
# 进行K折交叉验证，
def cross_validate_model(X, y, model_name: str, dataset_name: str, target: str):
    """
    执行k折交叉验证，根据训练数据集自身进行工作
    
    Returns:
        dict: CV结果
    """
    # 自动调整fold数（小数据集）
    min_class_count = np.bincount(y).min()
    n_folds = min(CONFIG['n_folds'], min_class_count)
    if n_folds < CONFIG['n_folds']:
        print(f"    ⚠ 样本数较少，调整fold数为 {n_folds}")
    
    if n_folds < 2:
        print("    ⚠ 样本过少，无法进行分层交叉验证，跳过该任务")
        return None

    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=CONFIG['random_state'])
    
    metrics = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'auc': []
    }

    from sklearn.utils import shuffle
    X, y = shuffle(X, y, random_state=random.randint(1, 1000000))
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # 训练模型
        model = get_model(model_name, CONFIG['use_gpu'])
        model.fit(X_train, y_train) # 这个地方是训练模型的核心代码
        
        # 预测
        y_pred = model.predict(X_test)
        # （修改后的probo）
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            y_proba = proba[:, 1]
        else:
            y_proba = None  # 只有一个类别时，AUC 没有定义
        
        # 计算指标
        metrics['accuracy'].append(accuracy_score(y_test, y_pred))
        metrics['precision'].append(precision_score(y_test, y_pred, average='binary', zero_division=0))
        metrics['recall'].append(recall_score(y_test, y_pred, average='binary', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='binary', zero_division=0))
        
        # AUC（需要至少两个类别，新增一种情况与下面配合，就是只有一种类别）
        if y_proba is not None and len(np.unique(y_test)) > 1:
            metrics['auc'].append(roc_auc_score(y_test, y_proba))
        else:
            metrics['auc'].append(np.nan)
    
    # 汇总结果
    results = {
        'dataset': f'cross_{dataset_name}',
        'target': target,
        'model': model_name,
        'n_folds': n_folds,
        'metrics': metrics,
        'mean_metrics': {k: np.nanmean(v) for k, v in metrics.items()},
        'std_metrics': {k: np.nanstd(v) for k, v in metrics.items()},
    }
    
    # 单独一次的结果为
    metric={k: np.nanmean(v) for k, v in metrics.items()}
    print("输出点啥",metric)
    return metric, results
# 进行独立验证的函数
def independent_validate(X_train, y_train, X_test, y_test, model_name: str, 
                         dataset_train_name: str, 
                         dataset_test_name: str,
                         target: str):
    """
    在训练集上进行全量训练，在独立的测试集上进行单次验证
    
    Returns:
        dict: 评估结果（结构与原CV结果一致，方便兼容）
    """
    # 检查类别情况，防止只有单类别无法计算指标
    unique_classes = np.unique(y_test)
    if len(unique_classes) < 2:
        print(f"    ⚠ 测试集只有类别 {unique_classes}，无法进行二分类评估，跳过该任务")
        return None

    # --- 核心训练过程 ---
    # 0. 刷新数据顺序
    from sklearn.utils import shuffle
    X_train, y_train = shuffle(X_train, y_train, random_state=random.randint(1, 1000000))
    # 1. 创建模型
    model = get_model(model_name, CONFIG['use_gpu'])
    # 2. 直接在全量 X_train 上训练，不再有 Fold 迭代
    model.fit(X_train, y_train) 

    
    
    # --- 预测过程 ---
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    
    if proba.shape[1] == 2:
        y_proba = proba[:, 1]
    else:
        y_proba = None

    # --- 计算指标 ---
    # 为了保持输出结构一致，我们将结果存入列表（虽然只有一个值）
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='binary', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='binary', zero_division=0),
        'f1': f1_score(y_test, y_pred, average='binary', zero_division=0),
        'auc': 0
    }
    
    if y_proba is not None and len(unique_classes) > 1:
        metrics['auc']=roc_auc_score(y_test, y_proba)
    else:
        metrics['auc']=np.nan

    # --- 汇总结果（保持与原 CV 格式兼容） ---
    results = {
        'dataset': f'{dataset_train_name}_{dataset_test_name}',
        'target': target,
        'model': model_name,
        'fold': 1, #  K-Fold
        'metrics': metrics,
        'mean_metrics': {k: np.nanmean(v) for k, v in metrics.items()}, # 均值即本身
        'std_metrics': {k: 0.0 for k in metrics.keys()},                # 单次验证无标准差
    }
    
    return metrics,results




## 5.重复运行，统计结果

注意一下执行的是啥东西。。。。

In [18]:
import matplotlib.pyplot as plt
import numpy as np

# 配置
po=1
N_RUNS = 2
METRIC_KEYS = ['accuracy', 'precision', 'recall', 'f1', 'auc']
MODELS = CONFIG['models_to_train']
TARGETS = ['SIF', 'SGF']

# 初始化容器：用于存储每一轮的均值结果
# 结构：storage[验证类型][target_model][metric_name] = [value1, value2, ...]
storage = {
    'num':N_RUNS,
    'cv': {
        'metrics':{
            'lr':[],
            'rf':[],
            'xgb':[],
        },
        'means':{
            'lr':[],
            'rf':[],
            'xgb':[],
        }
    },
    'ind': {
        'metrics':{
            'lr':[],
            'rf':[],
            'xgb':[],
        },
        'means':{
            'lr':[],
            'rf':[],
            'xgb':[],
        }

    }
}

# 首先，在循环开始前加载特征掩码文件
try:
    with open(CONFIG['temp']/CONFIG['mask'], "r") as f:
        mask_data = json.load(f)
    mask_arrays = mask_data.get("mask_arrays", {})
    print("✓ 特征掩码文件加载成功")
except FileNotFoundError:
    print("⚠ 未找到 importance_mask.json 文件，将继续使用所有特征")
    mask_arrays = {}

print(f"🚀 开始大规模验证（共 {N_RUNS} 轮迭代）...")
# --- 独立验证的循环 ---
for run in range(1, N_RUNS + 1):
    # 核心：动态修改随机种子，确保每一轮的划分和模型初始化都不同
    train_name=CONFIG['dataset_names']['train']
    test_name=CONFIG['dataset_names']['test']
    train_data = datasets_data.get(train_name)
    test_data = datasets_data.get(test_name)
    if train_data and test_data:
        target=CONFIG['Target']
        # 寻找importance_mask.json，
        # 里面有一个mask_arrays,是个数组，里面对应的key代表情况，
        # 0代表整个特征需要被放弃，1则代表这个特征需要保留
        X_tr, y_tr = train_data[f'X_{target.lower()}'], train_data[f'y_{target.lower()}']
        X_te, y_te = test_data[f'X_{target.lower()}'], test_data[f'y_{target.lower()}']
        
        for model_name in MODELS:
            task_key = f"{target.upper()}_{model_name.lower()}"  # 例如: "SIF_lr"
            # 设置预备数值
            X_tr_=X_tr
            X_te_=X_te
            
            # 检查是否存在该任务对应的掩码
            if task_key in mask_arrays:
                mask = mask_arrays[task_key]  # 这是一个由0和1组成的列表
                
                # 确保掩码长度与特征数一致
                if len(mask) == X_tr.shape[1]:
                    # 应用掩码：只保留掩码为1的特征列
                    feature_indices_to_keep = [i for i, val in enumerate(mask) if val == 1]
                    
                    if len(feature_indices_to_keep) > 0:
                        X_tr_ = X_tr[:, feature_indices_to_keep]
                        X_te_ = X_te[:, feature_indices_to_keep]
                        print(f"   应用特征掩码 [{task_key}]：保留 {len(feature_indices_to_keep)}/{len(mask)} 个特征")
                    else:
                        print(f"   警告：掩码 [{task_key}] 过滤后无特征保留，跳过此任务")
                        continue
                else:
                    print(f"   警告：掩码长度与特征数不符 [{task_key}]， 特征数目为{X_tr.shape[1]}， mask数目为{len(mask)}，跳过掩码过滤")
            else:
                print(f"   未找到任务 [{task_key}] 的掩码，使用所有特征")
            # --- 新增代码结束 ---

            try:
                # 注意：这里填入的是修改后的
                metric,results = independent_validate(X_tr_, y_tr, X_te_, y_te, model_name, train_name, test_name, target)
                if metric:
                    storage['ind']['metrics'][f"{model_name}"].append(metric)
            
            except Exception as e: 
                print(f"  ⚠ 独立验证出错: {e}")
                continue
    
    if run % 5 == 0:
        print(f"已独立验证的完成 {run}/{N_RUNS} 轮迭代...")
# --- 交叉验证的循环 ---
for run in range(1, N_RUNS + 1):
    dataset_name=TRAIN_NAME
    data=datasets_data[TRAIN_NAME]
    target=CONFIG['Target']
    X, y = data[f'X_{target.lower()}'], data[f'y_{target.lower()}']
    if len(y) == 0: continue
    for model_name in MODELS:
        try:
            met,  res = cross_validate_model(X, y, model_name, dataset_name, target)
            if met:
                storage['cv']['metrics'][f"{model_name}"].append(met)
        except: continue

print(storage)
import json
import numpy as np

output_json_path = CONFIG['Final_Results_json_path'] 
final_json_data = {}

for mode in ['cv','ind']:
    for model_name in MODELS:
        dists=storage[mode]['metrics'][f"{model_name}"]
        mean_dist = { k: round(float(np.nanmean([d[k] for d in dists])), 4)for k in dists[0]}
        storage[mode]['means'][f"{model_name}"]= mean_dist


try:
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(storage, f, indent=4, ensure_ascii=False)
    print(f"\n[成功] 评估结果已保存至: {output_json_path}")
except Exception as e:
    print(f"\n[失败] 保存 JSON 时出错: {e}")



⚠ 未找到 importance_mask.json 文件，将继续使用所有特征
🚀 开始大规模验证（共 2 轮迭代）...
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
输出点啥 {'accuracy': np.float64(0.8864864864864865), 'precision': np.float64(0.781923273657289), 'recall': np.float64(0.8209523809523809), 'f1': np.float64(0.7963167952641637), 'auc': np.float64(0.9487216731556354)}
输出点啥 {'accuracy': np.float64(0.8783783783783784), 'precision': np.float64(0.7911090225563909), 'recall': np.float64(0.7728571428571428), 'f1': np.float64(0.7775604224384713), 'auc': np.float64(0.9483815014475392)}
输出点啥 {'accuracy': np.float64(0.8756756756756756), 'precision': np.float64(0.7761487050960735), 'recall': np.float64(0.7619047619047619), 'f1': np.float64(0.7684466795506589), 'auc': np.float64(0.9362773285414796)}
输出点啥 {'accuracy': np.float64(0.8675675675675676), 'precision': np.float64(0.7394112554112555), 'recall': np.float64(0.822